# Note

This notebook needs the following zip file containing the GraphBlas backendto run.

[gbtl_src.zip](https://github.com/mattnkrueger/gbtl_src) (link to github page)

--

**Environments**
- `CPU`, Python3 (GBlas & OpenMP)
- `T4 GPU`, Python3 (CUDA)

In [ ]:
# unzip gblas repo
!unzip -oq gbtl_src.zip -d /content/
!g++ -std=c++17 -O3 \
    -I /content/gbtl_src \
    -I /content/gbtl_src/graphblas/platforms/optimized_sequential \
    sssp_delta_step_gbtl.cpp -o sssp_delta_step_gbtl

## 1) Sequential GBlas - `sssp_delta_step_gbtl.cpp` 

In [ ]:
%%writefile sssp_delta_step_gbtl.cpp
#include <chrono>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <iostream>
#include <vector>

#include <graphblas/graphblas.hpp>
#include <algorithms/sssp.hpp>

using Weight = unsigned int;

struct GnpParams {
    int avg_degree;
    int w_light;
    int w_heavy;
    int p_light_pct;
    unsigned long long seed;
};

static unsigned long long g_rng = 0x9E3779B97F4A7C15ULL;
static void rng_seed(unsigned long long s) {
    g_rng = s ? s : 0x9E3779B97F4A7C15ULL;
}
static unsigned long long rng_u64() {
    unsigned long long z = (g_rng += 0x9E3779B97F4A7C15ULL);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}
static double rng_unit() {
    return (double)(rng_u64() >> 11) * (1.0 / 9007199254740992.0);
}
static int rng_int_incl(int lo, int hi) {
    unsigned long long span = (unsigned long long)(hi - lo + 1);
    return lo + (int)(rng_u64() % span);
}
static Weight gnp_weight(GnpParams const &p) {
    int pct = (int)(rng_u64() % 100ULL);
    if (pct < p.p_light_pct) return (Weight)rng_int_incl(1, p.w_light);
    return (Weight)rng_int_incl(p.w_light + 1, p.w_heavy);
}
static Weight gnp_light_weight(GnpParams const &p) {
    return (Weight)rng_int_incl(1, p.w_light);
}
static void gnp_shuffle(std::vector<int> &a) {
    for (int i = (int)a.size() - 1; i > 0; --i) {
        int j = (int)(rng_u64() % (unsigned long long)(i + 1));
        std::swap(a[i], a[j]);
    }
}

struct EdgeRec { int v; Weight w; };

static void generateGnpGraph(int n,
                             GnpParams const &P,
                             std::vector<grb::IndexType> &rows,
                             std::vector<grb::IndexType> &cols,
                             std::vector<Weight>         &vals,
                             std::vector<std::vector<EdgeRec>> &adj)
{
    rng_seed(P.seed);

    auto add_undirected = [&](int u, int v, Weight w) {
        rows.push_back(u); cols.push_back(v); vals.push_back(w);
        rows.push_back(v); cols.push_back(u); vals.push_back(w);
        adj[u].push_back({v, w});
        adj[v].push_back({u, w});
    };

    std::vector<int> perm(n);
    for (int i = 0; i < n; ++i) perm[i] = i;
    gnp_shuffle(perm);
    for (int i = 1; i < n; ++i)
        add_undirected(perm[i - 1], perm[i], gnp_light_weight(P));

    if (n < 2 || P.avg_degree <= 0) return;

    double p = (double)P.avg_degree / (double)(n - 1);
    if (p >= 1.0) p = 0.999999;
    double lp = std::log(1.0 - p);

    int u = 1, v = -1;
    while (u < n) {
        double r = rng_unit();
        if (r <= 0.0) r = 1e-300;
        long long skip = 1 + (long long)std::floor(std::log(r) / lp);
        v += (int)skip;
        while (v >= u && u < n) { v -= u; ++u; }
        if (u >= n) break;
        Weight w = gnp_weight(P);
        add_undirected(v, u, w);
    }
}

static std::vector<int> reconstructPath(
    int src, int target,
    grb::Vector<Weight> const &paths,
    std::vector<std::vector<EdgeRec>> const &adj)
{
    std::vector<int> rev;
    if (!paths.hasElement(target)) return rev;
    int current = target;
    rev.push_back(current);
    while (current != src) {
        Weight dc = paths.extractElement(current);
        bool found = false;
        for (auto const &e : adj[current]) {
            if (!paths.hasElement(e.v)) continue;
            Weight dp = paths.extractElement(e.v);
            if (dp + e.w == dc) {
                current = e.v;
                rev.push_back(current);
                found = true;
                break;
            }
        }
        if (!found) { rev.clear(); return rev; }
    }
    return std::vector<int>(rev.rbegin(), rev.rend());
}

int main()
{
    int n, source, target;
    long long m_extra;
    int avg_degree_c, w_light, w_heavy, p_light_pct;
    unsigned long long seed;
    Weight delta;

    std::printf("=======================================================\n");
    std::printf("  DELTA-STEPPING SSSP (Sequential GBlas / GBTL)\n");
    std::printf("=======================================================\n\n");

    std::printf("Enter number of nodes: ");
    if (std::scanf("%d", &n) != 1) return 1;
    if (n <= 1) { std::printf("Number of nodes must be at least 2.\n"); return 1; }

    std::printf("Enter source node index (0 to %d): ", n - 1);
    if (std::scanf("%d", &source) != 1) return 1;
    std::printf("Enter target node index (0 to %d): ", n - 1);
    if (std::scanf("%d", &target) != 1) return 1;
    if (source < 0 || source >= n || target < 0 || target >= n) {
        std::printf("Invalid source or target index.\n"); return 1;
    }

    std::printf("Enter m_extra (ignored): ");
    if (std::scanf("%lld", &m_extra) != 1) return 1;
    (void)m_extra;

    std::printf("Enter the Delta Value: ");
    unsigned int delta_in;
    if (std::scanf("%u", &delta_in) != 1) return 1;
    if (delta_in == 0u) { std::printf("Delta must be > 0.\n"); return 1; }
    delta = (Weight)delta_in;

    std::printf("Enter avg_degree_c: ");
    if (std::scanf("%d", &avg_degree_c) != 1) return 1;
    std::printf("Enter W_LIGHT: ");
    if (std::scanf("%d", &w_light) != 1) return 1;
    std::printf("Enter W_HEAVY: ");
    if (std::scanf("%d", &w_heavy) != 1) return 1;
    std::printf("Enter p_light_pct: ");
    if (std::scanf("%d", &p_light_pct) != 1) return 1;
    std::printf("Enter seed: ");
    if (std::scanf("%llu", &seed) != 1) return 1;

    if (avg_degree_c < 0 || w_light < 1 || w_heavy <= w_light ||
        p_light_pct < 0 || p_light_pct > 100) {
        std::printf("Bad GNP params.\n"); return 1;
    }

    GnpParams P{avg_degree_c, w_light, w_heavy, p_light_pct, seed};

    std::printf("\n--- Building random graph ---\n");
    std::printf("Nodes: %d   Source: %d   Target: %d   Delta: %u   Seed: %llu\n",
                n, source, target, delta_in, seed);

    std::vector<grb::IndexType> rows, cols;
    std::vector<Weight>         vals;
    std::vector<std::vector<EdgeRec>> adj(n);
    long long approx_edges = ((long long)(n - 1) + (long long)avg_degree_c * (long long)n / 2 + 16) * 2;
    rows.reserve(approx_edges);
    cols.reserve(approx_edges);
    vals.reserve(approx_edges);

    generateGnpGraph(n, P, rows, cols, vals, adj);

    std::printf("Graph generation complete.\n");
    std::printf("Nodes: %d\n", n);
    std::printf("Directed edges stored: %zu\n", rows.size());
    std::printf("Undirected edges stored: %zu\n\n", rows.size() / 2);

    grb::Matrix<Weight> A((grb::IndexType)n, (grb::IndexType)n);
    A.build(rows.begin(), cols.begin(), vals.begin(),
            (grb::IndexType)rows.size(), grb::Min<Weight>());

    grb::Vector<Weight> paths((grb::IndexType)n);

    std::printf("Running Delta-Stepping SSSP (Sequential GBlas)...\n");
    auto t0 = std::chrono::high_resolution_clock::now();
    algorithms::sssp_delta_step(A, delta, (grb::IndexType)source, paths);
    auto t1 = std::chrono::high_resolution_clock::now();

    double runtime_ms = std::chrono::duration<double>(t1 - t0).count() * 1000.0;

    std::printf("\n=======================================================\n");
    std::printf("       SSSP RUNTIME (delta = %u)\n", delta_in);
    std::printf("=======================================================\n");
    std::printf(" %-32s | %-12s\n", "Algorithm", "Time (ms)");
    std::printf("-------------------------------------------------------\n");
    std::printf(" %-32s | %12.4f\n", "Delta-Stepping (Seq GBlas)", runtime_ms);
    std::printf("=======================================================\n\n");

    if (!paths.hasElement((grb::IndexType)target)) {
        std::printf("No path found from %d to %d.\n", source, target);
        return 0;
    }

    Weight cost = paths.extractElement((grb::IndexType)target);
    std::printf("Source: %d   Target: %d   Cost: %u\n", source, target, cost);

    auto path = reconstructPath(source, target, paths, adj);
    std::printf("Path: ");
    if (path.empty()) std::printf("(reconstruction failed)");
    else for (int v : path) std::printf("%d ", v);
    std::printf("\n");
    return 0;
}


Overwriting new_delta_step_gbtl.cpp


## 2) OpenMP - `sssp_delta_step_omp.cpp`

In [ ]:
%%writefile sssp_delta_step_omp.cpp
#include <atomic>
#include <chrono>
#include <climits>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <iostream>
#include <vector>

#include <omp.h>

using Weight = unsigned int;
static const Weight   INF_W      = 0xFFFFFFFFu;
static const uint32_t INF_BUCKET = 0xFFFFFFFFu;


struct GnpParams {
    int avg_degree;
    int w_light;
    int w_heavy;
    int p_light_pct;
    unsigned long long seed;
};

static unsigned long long g_rng = 0x9E3779B97F4A7C15ULL;
static void rng_seed(unsigned long long s) {
    g_rng = s ? s : 0x9E3779B97F4A7C15ULL;
}
static unsigned long long rng_u64() {
    unsigned long long z = (g_rng += 0x9E3779B97F4A7C15ULL);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}
static double rng_unit() {
    return (double)(rng_u64() >> 11) * (1.0 / 9007199254740992.0);
}
static int rng_int_incl(int lo, int hi) {
    unsigned long long span = (unsigned long long)(hi - lo + 1);
    return lo + (int)(rng_u64() % span);
}
static Weight gnp_weight(GnpParams const &p) {
    int pct = (int)(rng_u64() % 100ULL);
    if (pct < p.p_light_pct) return (Weight)rng_int_incl(1, p.w_light);
    return (Weight)rng_int_incl(p.w_light + 1, p.w_heavy);
}
static Weight gnp_light_weight(GnpParams const &p) {
    return (Weight)rng_int_incl(1, p.w_light);
}
static void gnp_shuffle(std::vector<int> &a) {
    for (int i = (int)a.size() - 1; i > 0; --i) {
        int j = (int)(rng_u64() % (unsigned long long)(i + 1));
        std::swap(a[i], a[j]);
    }
}

struct EdgeRec { int v; Weight w; };

static void generateGnpGraph(int n,
                             GnpParams const &P,
                             std::vector<std::vector<EdgeRec>> &adj,
                             long long &dir_edges)
{
    rng_seed(P.seed);

    auto add_undirected = [&](int u, int v, Weight w) {
        adj[u].push_back({v, w});
        adj[v].push_back({u, w});
        dir_edges += 2;
    };

    std::vector<int> perm(n);
    for (int i = 0; i < n; ++i) perm[i] = i;
    gnp_shuffle(perm);
    for (int i = 1; i < n; ++i)
        add_undirected(perm[i - 1], perm[i], gnp_light_weight(P));

    if (n < 2 || P.avg_degree <= 0) return;

    double p = (double)P.avg_degree / (double)(n - 1);
    if (p >= 1.0) p = 0.999999;
    double lp = std::log(1.0 - p);

    int u = 1, v = -1;
    while (u < n) {
        double r = rng_unit();
        if (r <= 0.0) r = 1e-300;
        long long skip = 1 + (long long)std::floor(std::log(r) / lp);
        v += (int)skip;
        while (v >= u && u < n) { v -= u; ++u; }
        if (u >= n) break;
        Weight w = gnp_weight(P);
        add_undirected(v, u, w);
    }
}


// ============================================================================
// CSR with per-vertex light/heavy partition.
// ============================================================================
struct CSR {
    int n = 0;
    std::vector<long long> row_off;
    std::vector<long long> light_end;
    std::vector<int>       col_idx;
    std::vector<Weight>    weights;
};

static void build_csr_split(int n,
                            std::vector<std::vector<EdgeRec>> const &adj,
                            Weight delta,
                            CSR &G)
{
    G.n = n;
    G.row_off.assign(n + 1, 0);
    G.light_end.assign(n, 0);
    long long m = 0;
    for (int u = 0; u < n; ++u) m += (long long)adj[u].size();
    G.col_idx.resize(m);
    G.weights.resize(m);
    long long off = 0;
    for (int u = 0; u < n; ++u) {
        G.row_off[u] = off;
        for (auto const &e : adj[u]) if (e.w <= delta) {
            G.col_idx[off] = e.v; G.weights[off] = e.w; ++off;
        }
        G.light_end[u] = off;
        for (auto const &e : adj[u]) if (e.w >  delta) {
            G.col_idx[off] = e.v; G.weights[off] = e.w; ++off;
        }
    }
    G.row_off[n] = off;
}

static inline bool relax_tent(Weight *tent, uint32_t *bidx, int v, Weight new_dist, Weight delta) {
    Weight old = __atomic_load_n(&tent[v], __ATOMIC_ACQUIRE);
    while (new_dist < old) {
        if (__atomic_compare_exchange_n(&tent[v], &old, new_dist,
                                        false, __ATOMIC_ACQ_REL, __ATOMIC_ACQUIRE)) {
            uint32_t nb = (uint32_t)(new_dist / delta);
            __atomic_store_n(&bidx[v], nb, __ATOMIC_RELEASE);
            return true;
        }
    }
    return false;
}

static void omp_delta_step(int n, CSR const &G, int source, Weight delta,
                           std::vector<Weight> &tent)
{
    tent.assign(n, INF_W);
    std::vector<uint32_t> bidx(n, INF_BUCKET);
    tent[source] = 0;
    bidx[source] = 0;

    int max_threads = omp_get_max_threads();
    std::vector<std::vector<int>> S_local(max_threads);
    std::vector<char> in_R(n, 0);
    std::vector<int>  R, S;
    uint32_t cur_bucket = 0;

    const int PAR_N_THRESH = 4096;
    const int PAR_S_THRESH = 256;

    while (true) {
        uint32_t min_b = INF_BUCKET;
        #pragma omp parallel for reduction(min:min_b) schedule(static) \
            if(n >= PAR_N_THRESH)
        for (int v = 0; v < n; ++v) {
            uint32_t b = bidx[v];
            if (b != INF_BUCKET && b >= cur_bucket && b < min_b) min_b = b;
        }
        if (min_b == INF_BUCKET) break;
        cur_bucket = min_b;

        std::fill(in_R.begin(), in_R.end(), 0);

        while (true) {
            for (auto &sl : S_local) sl.clear();
            #pragma omp parallel if(n >= PAR_N_THRESH)
            {
                int tid = omp_get_thread_num();
                auto &sl = S_local[tid];
                #pragma omp for nowait schedule(static)
                for (int v = 0; v < n; ++v)
                    if (bidx[v] == cur_bucket) sl.push_back(v);
            }
            size_t total = 0;
            for (auto &sl : S_local) total += sl.size();
            if (total == 0) break;

            S.clear();
            S.reserve(total);
            for (auto &sl : S_local) S.insert(S.end(), sl.begin(), sl.end());

            int S_size = (int)S.size();

            #pragma omp parallel for schedule(static) if(S_size >= PAR_S_THRESH)
            for (int i = 0; i < S_size; ++i) {
                int u = S[i];
                in_R[u] = 1;
                __atomic_store_n(&bidx[u], INF_BUCKET, __ATOMIC_RELEASE);
            }

            #pragma omp parallel for schedule(dynamic, 128) if(S_size >= PAR_S_THRESH)
            for (int i = 0; i < S_size; ++i) {
                int u  = S[i];
                Weight tu = __atomic_load_n(&tent[u], __ATOMIC_ACQUIRE);
                if (tu == INF_W) continue;
                long long s = G.row_off[u];
                long long e = G.light_end[u];
                for (long long k = s; k < e; ++k) {
                    int    w = G.col_idx[k];
                    Weight c = G.weights[k];
                    Weight nd = tu + c;
                    if (nd < tu) continue;
                    relax_tent(tent.data(), bidx.data(), w, nd, delta);
                }
            }
        }

        R.clear();
        for (int v = 0; v < n; ++v) if (in_R[v]) R.push_back(v);
        int R_size = (int)R.size();

        #pragma omp parallel for schedule(dynamic, 128) if(R_size >= PAR_S_THRESH)
        for (int i = 0; i < R_size; ++i) {
            int u = R[i];
            Weight tu = __atomic_load_n(&tent[u], __ATOMIC_ACQUIRE);
            if (tu == INF_W) continue;
            long long s = G.light_end[u];
            long long e = G.row_off[u + 1];
            for (long long k = s; k < e; ++k) {
                int    w = G.col_idx[k];
                Weight c = G.weights[k];
                Weight nd = tu + c;
                if (nd < tu) continue;
                relax_tent(tent.data(), bidx.data(), w, nd, delta);
            }
        }

        ++cur_bucket;
    }
}

static std::vector<int> reconstructPath(
    int src, int target,
    std::vector<Weight> const &tent,
    std::vector<std::vector<EdgeRec>> const &adj)
{
    std::vector<int> rev;
    if (target < 0 || (size_t)target >= tent.size()) return rev;
    if (tent[target] == INF_W) return rev;
    int current = target;
    rev.push_back(current);
    while (current != src) {
        Weight dc = tent[current];
        bool found = false;
        for (auto const &e : adj[current]) {
            if (tent[e.v] == INF_W) continue;
            if ((Weight)(tent[e.v] + e.w) == dc) {
                current = e.v;
                rev.push_back(current);
                found = true;
                break;
            }
        }
        if (!found) { rev.clear(); return rev; }
    }
    return std::vector<int>(rev.rbegin(), rev.rend());
}

int main()
{
    int n, source, target;
    long long m_extra;
    int avg_degree_c, w_light, w_heavy, p_light_pct;
    unsigned long long seed;
    Weight delta;

    std::printf("=======================================================\n");
    std::printf("  DELTA-STEPPING SSSP (OpenMP)\n");
    std::printf("=======================================================\n\n");

    std::printf("Enter number of nodes: ");
    if (std::scanf("%d", &n) != 1) return 1;
    if (n <= 1) { std::printf("Number of nodes must be at least 2.\n"); return 1; }

    std::printf("Enter source node index (0 to %d): ", n - 1);
    if (std::scanf("%d", &source) != 1) return 1;
    std::printf("Enter target node index (0 to %d): ", n - 1);
    if (std::scanf("%d", &target) != 1) return 1;
    if (source < 0 || source >= n || target < 0 || target >= n) {
        std::printf("Invalid source or target index.\n"); return 1;
    }

    std::printf("Enter m_extra (ignored): ");
    if (std::scanf("%lld", &m_extra) != 1) return 1;
    (void)m_extra;

    std::printf("Enter the Delta Value: ");
    unsigned int delta_in;
    if (std::scanf("%u", &delta_in) != 1) return 1;
    if (delta_in == 0u) { std::printf("Delta must be > 0.\n"); return 1; }
    delta = (Weight)delta_in;

    std::printf("Enter avg_degree_c: ");
    if (std::scanf("%d", &avg_degree_c) != 1) return 1;
    std::printf("Enter W_LIGHT: ");
    if (std::scanf("%d", &w_light) != 1) return 1;
    std::printf("Enter W_HEAVY: ");
    if (std::scanf("%d", &w_heavy) != 1) return 1;
    std::printf("Enter p_light_pct: ");
    if (std::scanf("%d", &p_light_pct) != 1) return 1;
    std::printf("Enter seed: ");
    if (std::scanf("%llu", &seed) != 1) return 1;

    if (avg_degree_c < 0 || w_light < 1 || w_heavy <= w_light ||
        p_light_pct < 0 || p_light_pct > 100) {
        std::printf("Bad GNP params.\n"); return 1;
    }

    GnpParams P{avg_degree_c, w_light, w_heavy, p_light_pct, seed};

    std::printf("\n--- Building random graph ---\n");
    std::printf("Nodes: %d   Source: %d   Target: %d   Delta: %u   Seed: %llu\n",
                n, source, target, delta_in, seed);

    std::vector<std::vector<EdgeRec>> adj(n);
    long long dir_edges = 0;
    generateGnpGraph(n, P, adj, dir_edges);

    std::printf("Graph generation complete.\n");
    std::printf("Nodes: %d\n", n);
    std::printf("Directed edges stored: %lld\n", dir_edges);
    std::printf("Undirected edges stored: %lld\n", dir_edges / 2);
    std::printf("OpenMP threads: %d\n\n", omp_get_max_threads());

    CSR G;
    build_csr_split(n, adj, delta, G);

    std::vector<Weight> tent;

    std::printf("Running Delta-Stepping SSSP (OpenMP)...\n");
    auto t0 = std::chrono::high_resolution_clock::now();
    omp_delta_step(n, G, source, delta, tent);
    auto t1 = std::chrono::high_resolution_clock::now();

    double runtime_ms = std::chrono::duration<double>(t1 - t0).count() * 1000.0;

    std::printf("\n=======================================================\n");
    std::printf("       SSSP RUNTIME (delta = %u)\n", delta_in);
    std::printf("=======================================================\n");
    std::printf(" %-32s | %-12s\n", "Algorithm", "Time (ms)");
    std::printf("-------------------------------------------------------\n");
    std::printf(" %-32s | %12.4f\n", "Delta-Stepping (OpenMP)", runtime_ms);
    std::printf("=======================================================\n\n");

    if (tent[target] == INF_W) {
        std::printf("No path found from %d to %d.\n", source, target);
        return 0;
    }

    std::printf("Source: %d   Target: %d   Cost: %u\n", source, target, tent[target]);

    auto path = reconstructPath(source, target, tent, adj);
    std::printf("Path: ");
    if (path.empty()) std::printf("(reconstruction failed)");
    else for (int v : path) std::printf("%d ", v);
    std::printf("\n");

    return 0;
}


Writing new_delta_step_omp.cpp


In [ ]:
!g++ -std=c++17 -O3 -fopenmp sssp_delta_step_omp.cpp -o sssp_delta_step_omp

## 3) CUDA - `sssp_delta_step_cuda.cu`

In [ ]:
%%writefile sssp_delta_step_cuda.cu
#include <cuda_runtime.h>
#include <limits.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define CUDA_CHECK(call)                                                       \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(err));      \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

typedef unsigned int Weight;
static const Weight INF_W = 0xFFFFFFFFu;

typedef struct { int to, weight, next; } Edge;
typedef struct { int n, m, capacity; int *head; Edge *edges; } Graph;

Graph *createGraph(int n, long long capacity) {
    Graph *g = (Graph *)malloc(sizeof(Graph));
    g->n = n;
    g->m = 0;
    long long cap = 2LL * capacity;
    if (cap < 4) cap = 4;
    g->capacity = (int)cap;
    g->head = (int *)malloc(n * sizeof(int));
    g->edges = (Edge *)malloc(g->capacity * sizeof(Edge));
    for (int i = 0; i < n; i++) g->head[i] = -1;
    return g;
}

void addEdge(Graph *g, int u, int v, int w) {
    if (g->m >= g->capacity) {
        long long nc = (long long)g->capacity * 2 + 16;
        Edge *ne = (Edge *)realloc(g->edges, (size_t)nc * sizeof(Edge));
        if (!ne) { fprintf(stderr, "edge resize fail.\n"); exit(1); }
        g->edges = ne;
        g->capacity = (int)nc;
    }
    g->edges[g->m].to = v;
    g->edges[g->m].weight = w;
    g->edges[g->m].next = g->head[u];
    g->head[u] = g->m++;
}

void addUndirectedEdge(Graph *g, int u, int v, int w) {
    addEdge(g, u, v, w);
    addEdge(g, v, u, w);
}

void freeGraph(Graph *g) {
    if (!g) return;
    free(g->head);
    free(g->edges);
    free(g);
}

typedef struct {
    int avg_degree;
    int w_light;
    int w_heavy;
    int p_light_pct;
    unsigned long long seed;
} GnpParams;

static unsigned long long g_rng = 0x9E3779B97F4A7C15ULL;
static void rng_seed(unsigned long long s) {
    g_rng = s ? s : 0x9E3779B97F4A7C15ULL;
}
static unsigned long long rng_u64(void) {
    unsigned long long z = (g_rng += 0x9E3779B97F4A7C15ULL);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}
static double rng_unit(void) {
    return (double)(rng_u64() >> 11) * (1.0 / 9007199254740992.0);
}
static int rng_int_incl(int lo, int hi) {
    unsigned long long span = (unsigned long long)(hi - lo + 1);
    return lo + (int)(rng_u64() % span);
}
static int gnp_weight(const GnpParams *p) {
    int pct = (int)(rng_u64() % 100ULL);
    if (pct < p->p_light_pct) return rng_int_incl(1, p->w_light);
    return rng_int_incl(p->w_light + 1, p->w_heavy);
}
static int gnp_light_weight(const GnpParams *p) {
    return rng_int_incl(1, p->w_light);
}
static void gnp_shuffle(int *a, int n) {
    for (int i = n - 1; i > 0; --i) {
        int j = (int)(rng_u64() % (unsigned long long)(i + 1));
        int t = a[i]; a[i] = a[j]; a[j] = t;
    }
}

static void generateGnpGraph(Graph *g, const GnpParams *P) {
    int n = g->n;
    rng_seed(P->seed);

    int *perm = (int *)malloc(n * sizeof(int));
    for (int i = 0; i < n; i++) perm[i] = i;
    gnp_shuffle(perm, n);
    for (int i = 1; i < n; i++)
        addUndirectedEdge(g, perm[i - 1], perm[i], gnp_light_weight(P));
    free(perm);

    if (n < 2 || P->avg_degree <= 0) return;

    double p = (double)P->avg_degree / (double)(n - 1);
    if (p >= 1.0) p = 0.999999;
    double lp = log(1.0 - p);

    int u = 1, v = -1;
    while (u < n) {
        double r = rng_unit();
        if (r <= 0.0) r = 1e-300;
        long long skip = 1 + (long long)floor(log(r) / lp);
        v += (int)skip;
        while (v >= u && u < n) { v -= u; u++; }
        if (u >= n) break;
        int w = gnp_weight(P);
        addUndirectedEdge(g, v, u, w);
    }
}

typedef struct { int v; Weight w; } EdgeRec;

static void buildAdj(Graph *g, EdgeRec ***outAdj, int **outDeg) {
    int n = g->n;
    int *deg = (int *)calloc(n, sizeof(int));
    for (int u = 0; u < n; u++)
        for (int e = g->head[u]; e != -1; e = g->edges[e].next) deg[u]++;
    EdgeRec **adj = (EdgeRec **)malloc(n * sizeof(EdgeRec *));
    for (int u = 0; u < n; u++) {
        adj[u] = (EdgeRec *)malloc((size_t)deg[u] * sizeof(EdgeRec));
        int k = 0;
        for (int e = g->head[u]; e != -1; e = g->edges[e].next) {
            adj[u][k].v = g->edges[e].to;
            adj[u][k].w = (Weight)g->edges[e].weight;
            k++;
        }
    }
    *outAdj = adj;
    *outDeg = deg;
}

static void freeAdj(int n, EdgeRec **adj, int *deg) {
    for (int i = 0; i < n; i++) free(adj[i]);
    free(adj);
    free(deg);
}

typedef struct {
    int *eu, *ev;
    Weight *ew;
    int nedges;
} EdgeList;

static void freeEdgeList(EdgeList *L) {
    free(L->eu); free(L->ev); free(L->ew);
}

static void splitLightHeavy(Graph *g, Weight delta, EdgeList *light, EdgeList *heavy) {
    int ec = 0;
    for (int u = 0; u < g->n; u++)
        for (int e = g->head[u]; e != -1; e = g->edges[e].next) ec++;
    light->eu = (int *)malloc(ec * sizeof(int));
    light->ev = (int *)malloc(ec * sizeof(int));
    light->ew = (Weight *)malloc(ec * sizeof(Weight));
    heavy->eu = (int *)malloc(ec * sizeof(int));
    heavy->ev = (int *)malloc(ec * sizeof(int));
    heavy->ew = (Weight *)malloc(ec * sizeof(Weight));
    light->nedges = 0;
    heavy->nedges = 0;
    for (int u = 0; u < g->n; u++) {
        for (int e = g->head[u]; e != -1; e = g->edges[e].next) {
            int v = g->edges[e].to;
            Weight w = (Weight)g->edges[e].weight;
            if (w <= delta) {
                int k = light->nedges++;
                light->eu[k] = u; light->ev[k] = v; light->ew[k] = w;
            } else {
                int k = heavy->nedges++;
                heavy->eu[k] = u; heavy->ev[k] = v; heavy->ew[k] = w;
            }
        }
    }
}

__global__ void k_init_dist(unsigned *d, int n, int src) {
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v >= n) return;
    d[v] = (v == src) ? 0u : INF_W;
}

__global__ void k_relax_edges(const int *eu, const int *ev, const Weight *ew, int ne,
                              const unsigned *dist_in, unsigned *dist_out,
                              const unsigned char *mask_u, Weight delta, int is_heavy) {
    int e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e >= ne) return;
    int u = eu[e];
    int v = ev[e];
    Weight w = ew[e];
    if (is_heavy) { if (w <= delta) return; }
    else          { if (w >  delta) return; }
    if (!mask_u[u]) return;
    unsigned du = dist_in[u];
    if (du == INF_W) return;
    unsigned long long sum = (unsigned long long)du + (unsigned long long)w;
    if (sum > (unsigned long long)INF_W - 1ULL) return;
    unsigned nd = (unsigned)sum;
    atomicMin((unsigned *)&dist_out[v], nd);
}

static int any_t_ge_i_delta(const unsigned *t, int n, unsigned long long i, Weight delta) {
    unsigned long long thr = i * (unsigned long long)delta;
    for (int v = 0; v < n; v++) {
        if (t[v] == INF_W) continue;
        if ((unsigned long long)t[v] >= thr) return 1;
    }
    return 0;
}

static void mask_in_bucket(const unsigned *t, int n, unsigned long long i, Weight delta,
                           unsigned char *mask) {
    unsigned long long lo = i * (unsigned long long)delta;
    unsigned long long hi = (i + 1ULL) * (unsigned long long)delta;
    for (int v = 0; v < n; v++) {
        if (t[v] == INF_W) { mask[v] = 0; continue; }
        unsigned long long tv = t[v];
        mask[v] = (tv >= lo && tv < hi) ? 1 : 0;
    }
}

static void mask_S_and_t(const unsigned *t, int n, const unsigned char *S, unsigned char *m) {
    for (int v = 0; v < n; v++)
        m[v] = (S[v] && t[v] != INF_W) ? 1 : 0;
}

static void delta_stepping_cuda(Graph *g, int src, Weight delta, unsigned *h_dist_out,
                                double *out_seconds) {
    int n = g->n;
    EdgeList Lh, Hh;
    splitLightHeavy(g, delta, &Lh, &Hh);

    int *d_leu, *d_lev;
    Weight *d_lew;
    int *d_heu, *d_hev;
    Weight *d_hew;
    CUDA_CHECK(cudaMalloc(&d_leu, sizeof(int) * (Lh.nedges > 0 ? Lh.nedges : 1)));
    CUDA_CHECK(cudaMalloc(&d_lev, sizeof(int) * (Lh.nedges > 0 ? Lh.nedges : 1)));
    CUDA_CHECK(cudaMalloc(&d_lew, sizeof(Weight) * (Lh.nedges > 0 ? Lh.nedges : 1)));
    CUDA_CHECK(cudaMalloc(&d_heu, sizeof(int) * (Hh.nedges > 0 ? Hh.nedges : 1)));
    CUDA_CHECK(cudaMalloc(&d_hev, sizeof(int) * (Hh.nedges > 0 ? Hh.nedges : 1)));
    CUDA_CHECK(cudaMalloc(&d_hew, sizeof(Weight) * (Hh.nedges > 0 ? Hh.nedges : 1)));
    if (Lh.nedges > 0) {
        CUDA_CHECK(cudaMemcpy(d_leu, Lh.eu, sizeof(int) * Lh.nedges, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_lev, Lh.ev, sizeof(int) * Lh.nedges, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_lew, Lh.ew, sizeof(Weight) * Lh.nedges, cudaMemcpyHostToDevice));
    }
    if (Hh.nedges > 0) {
        CUDA_CHECK(cudaMemcpy(d_heu, Hh.eu, sizeof(int) * Hh.nedges, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_hev, Hh.ev, sizeof(int) * Hh.nedges, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_hew, Hh.ew, sizeof(Weight) * Hh.nedges, cudaMemcpyHostToDevice));
    }

    unsigned *d0, *d1;
    unsigned char *d_mask;
    CUDA_CHECK(cudaMalloc(&d0, sizeof(unsigned) * n));
    CUDA_CHECK(cudaMalloc(&d1, sizeof(unsigned) * n));
    CUDA_CHECK(cudaMalloc(&d_mask, n));

    cudaEvent_t ev0, ev1;
    cudaEventCreate(&ev0);
    cudaEventCreate(&ev1);
    cudaEventRecord(ev0);

    int threads = 256;
    int blocks_n = (n + threads - 1) / threads;
    k_init_dist<<<blocks_n, threads>>>(d0, n, src);
    CUDA_CHECK(cudaDeviceSynchronize());

    unsigned *h_t = (unsigned *)malloc(sizeof(unsigned) * n);
    unsigned *h_prev = (unsigned *)malloc(sizeof(unsigned) * n);
    unsigned char *h_mask = (unsigned char *)malloc(n);
    unsigned char *h_S = (unsigned char *)calloc(n, 1);

    CUDA_CHECK(cudaMemcpy(h_t, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToHost));

    unsigned long long i_phase = 0;
    while (any_t_ge_i_delta(h_t, n, i_phase, delta)) {
        memset(h_S, 0, n);

        for (;;) {
            mask_in_bucket(h_t, n, i_phase, delta, h_mask);
            int any = 0;
            for (int k = 0; k < n; k++) if (h_mask[k]) { any = 1; break; }
            if (!any) break;

            for (int k = 0; k < n; k++) if (h_mask[k]) h_S[k] = 1;

            unsigned *snap = (unsigned *)malloc(sizeof(unsigned) * n);
            memcpy(snap, h_t, sizeof(unsigned) * n);
            memcpy(h_prev, h_t, sizeof(unsigned) * n);
            CUDA_CHECK(cudaMemcpy(d0, h_t, sizeof(unsigned) * n, cudaMemcpyHostToDevice));

            int inner_iters = 0;
            const int max_inner = n + 64;
            while (inner_iters < max_inner) {
                CUDA_CHECK(cudaMemcpy(d_mask, h_mask, n, cudaMemcpyHostToDevice));
                CUDA_CHECK(cudaMemcpy(d1, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
                int ne = Lh.nedges;
                if (ne > 0) {
                    int bl = (ne + threads - 1) / threads;
                    k_relax_edges<<<bl, threads>>>(d_leu, d_lev, d_lew, ne, d0, d1, d_mask, delta, 0);
                    CUDA_CHECK(cudaDeviceSynchronize());
                }
                CUDA_CHECK(cudaMemcpy(d0, d1, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
                CUDA_CHECK(cudaMemcpy(h_t, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToHost));

                int changed = 0;
                for (int k = 0; k < n; k++) if (h_t[k] != h_prev[k]) { changed = 1; break; }
                memcpy(h_prev, h_t, sizeof(unsigned) * n);
                mask_in_bucket(h_t, n, i_phase, delta, h_mask);
                inner_iters++;
                if (!changed) break;
            }

            int no_progress = 1;
            for (int k = 0; k < n; k++) if (h_t[k] != snap[k]) { no_progress = 0; break; }
            free(snap);
            if (no_progress) break;
        }

        mask_S_and_t(h_t, n, h_S, h_mask);
        CUDA_CHECK(cudaMemcpy(d_mask, h_mask, n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d0, h_t, sizeof(unsigned) * n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d1, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
        int neh = Hh.nedges;
        if (neh > 0) {
            int bh = (neh + threads - 1) / threads;
            k_relax_edges<<<bh, threads>>>(d_heu, d_hev, d_hew, neh, d0, d1, d_mask, delta, 1);
            CUDA_CHECK(cudaDeviceSynchronize());
        }
        CUDA_CHECK(cudaMemcpy(d0, d1, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
        CUDA_CHECK(cudaMemcpy(h_t, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToHost));

        i_phase++;
    }

    memcpy(h_dist_out, h_t, sizeof(unsigned) * n);

    cudaEventRecord(ev1);
    cudaEventSynchronize(ev1);
    float ms;
    cudaEventElapsedTime(&ms, ev0, ev1);
    *out_seconds = (double)ms / 1000.0;

    cudaEventDestroy(ev0);
    cudaEventDestroy(ev1);
    free(h_t); free(h_prev); free(h_mask); free(h_S);
    cudaFree(d0); cudaFree(d1); cudaFree(d_mask);
    cudaFree(d_leu); cudaFree(d_lev); cudaFree(d_lew);
    cudaFree(d_heu); cudaFree(d_hev); cudaFree(d_hew);
    freeEdgeList(&Lh);
    freeEdgeList(&Hh);
}

static void reconstructPath(int src, int target, const unsigned *dist, EdgeRec **adj, int *deg,
                            int *pathOut, int *pathLen) {
    *pathLen = 0;
    if (dist[target] == INF_W) return;
    int rev[4096];
    int rc = 0;
    int cur = target;
    rev[rc++] = cur;
    while (cur != src) {
        unsigned dc = dist[cur];
        int found = 0;
        for (int k = 0; k < deg[cur]; k++) {
            int p = adj[cur][k].v;
            Weight w = adj[cur][k].w;
            if (dist[p] == INF_W) continue;
            if ((unsigned long long)dist[p] + (unsigned long long)w == (unsigned long long)dc) {
                cur = p;
                rev[rc++] = cur;
                found = 1;
                break;
            }
        }
        if (!found) { *pathLen = 0; return; }
        if (rc >= 4096) { *pathLen = 0; return; }
    }
    *pathLen = rc;
    for (int i = 0; i < rc; i++) pathOut[i] = rev[rc - 1 - i];
}

int main(void) {
    int n, source, target;
    long long m_extra;
    unsigned int delta_in;
    int avg_degree_c, w_light, w_heavy, p_light_pct;
    unsigned long long seed;

    printf("=======================================================\n");
    printf("  DELTA-STEPPING SSSP (CUDA)\n");
    printf("=======================================================\n\n");

    int gpu_count = 0;
    if (cudaGetDeviceCount(&gpu_count) == cudaSuccess && gpu_count > 0) {
        cudaDeviceProp prop;
        if (cudaGetDeviceProperties(&prop, 0) == cudaSuccess) {
            printf("CUDA Device: %s (Compute Capability %d.%d)\n\n",
                   prop.name, prop.major, prop.minor);
        }
    }

    printf("Enter number of nodes: ");
    if (scanf("%d", &n) != 1) return 1;
    if (n <= 1) { printf("Number of nodes must be at least 2.\n"); return 1; }

    printf("Enter source node index (0 to %d): ", n - 1);
    if (scanf("%d", &source) != 1) return 1;
    printf("Enter target node index (0 to %d): ", n - 1);
    if (scanf("%d", &target) != 1) return 1;
    if (source < 0 || source >= n || target < 0 || target >= n) {
        printf("Invalid source or target index.\n"); return 1;
    }

    printf("Enter m_extra (ignored): ");
    if (scanf("%lld", &m_extra) != 1) return 1;
    (void)m_extra;

    printf("Enter the Delta Value: ");
    if (scanf("%u", &delta_in) != 1) return 1;
    if (delta_in == 0u) { printf("Delta must be > 0.\n"); return 1; }
    Weight delta = (Weight)delta_in;

    printf("Enter avg_degree_c: ");
    if (scanf("%d", &avg_degree_c) != 1) return 1;
    printf("Enter W_LIGHT: ");
    if (scanf("%d", &w_light) != 1) return 1;
    printf("Enter W_HEAVY: ");
    if (scanf("%d", &w_heavy) != 1) return 1;
    printf("Enter p_light_pct: ");
    if (scanf("%d", &p_light_pct) != 1) return 1;
    printf("Enter seed: ");
    if (scanf("%llu", &seed) != 1) return 1;

    if (avg_degree_c < 0 || w_light < 1 || w_heavy <= w_light ||
        p_light_pct < 0 || p_light_pct > 100) {
        printf("Bad GNP params.\n"); return 1;
    }

    GnpParams P;
    P.avg_degree = avg_degree_c;
    P.w_light = w_light;
    P.w_heavy = w_heavy;
    P.p_light_pct = p_light_pct;
    P.seed = seed;

    printf("\n--- Building random graph ---\n");
    printf("Nodes: %d   Source: %d   Target: %d   Delta: %u   Seed: %llu\n",
           n, source, target, delta_in, seed);

    long long expected_extra = (long long)avg_degree_c * (long long)n / 2;
    long long cap = (long long)(n - 1) + expected_extra + 1024;
    Graph *g = createGraph(n, cap);
    generateGnpGraph(g, &P);

    printf("Graph generation complete.\n");
    printf("Nodes: %d\n", n);
    printf("Directed edges stored: %d\n", g->m);
    printf("Undirected edges stored: %d\n\n", g->m / 2);

    EdgeRec **adj;
    int *deg;
    buildAdj(g, &adj, &deg);

    unsigned *dist = (unsigned *)malloc(sizeof(unsigned) * n);
    double runtime = 0.0;
    printf("Running Delta-Stepping SSSP (CUDA)...\n");
    delta_stepping_cuda(g, source, delta, dist, &runtime);

    double runtime_ms = runtime * 1000.0;

    printf("\n=======================================================\n");
    printf("       SSSP RUNTIME (delta = %u)\n", delta_in);
    printf("=======================================================\n");
    printf(" %-32s | %-12s\n", "Algorithm", "Time (ms)");
    printf("-------------------------------------------------------\n");
    printf(" %-32s | %12.4f\n", "Delta-Stepping (CUDA)", runtime_ms);
    printf("=======================================================\n\n");

    if (dist[target] == INF_W) {
        printf("No path found from %d to %d.\n", source, target);
        freeGraph(g); freeAdj(n, adj, deg); free(dist);
        return 0;
    }

    printf("Source: %d   Target: %d   Cost: %u\n", source, target, dist[target]);

    int path[4096];
    int plen;
    reconstructPath(source, target, dist, adj, deg, path, &plen);
    printf("Path: ");
    if (plen == 0) printf("(reconstruction failed)");
    else for (int i = 0; i < plen; i++) printf("%d ", path[i]);
    printf("\n");

    freeGraph(g);
    freeAdj(n, adj, deg);
    free(dist);
    return 0;
}


Overwriting new_delta_step_cuda.cu


In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 sssp_delta_step_cuda.cu -o sssp_delta_step_cuda